# 1) récupérer le code de nettoyage depuis GitHub

In [1]:
!git clone https://github.com/ellakonan41/llm_pii_anonymizer_fr.git
import sys
sys.path.append("llm_pii_anonymizer_fr")

from src.cleaning import normaliser_etiquettes, est_valide, retirer_fuite, tirer_echantillon

Cloning into 'llm_pii_anonymizer_fr'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 18 (delta 2), reused 18 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 25.31 KiB | 5.06 MiB/s, done.
Resolving deltas: 100% (2/2), done.


##  2) charger le français et garder les colonnes utiles

In [2]:
from datasets import load_dataset

ds = load_dataset("ai4privacy/open-pii-masking-500k-ai4privacy")
ds_fr = ds.filter(lambda ex: ex["language"] == "fr")

colonnes = ["uid", "source_text", "masked_text", "privacy_mask"]
train = ds_fr["train"].select_columns(colonnes).to_list()
validation = ds_fr["validation"].select_columns(colonnes).to_list()
print(len(train), len(validation))

README.md: 0.00B [00:00, ?B/s]

data/train/train.jsonl:   0%|          | 0.00/566M [00:00<?, ?B/s]

data/validation/test.jsonl:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/464150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/116077 [00:00<?, ? examples/s]

Filter:   0%|          | 0/464150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/116077 [00:00<?, ? examples/s]

89670 22466


## 3) appliquer le nettoyage minimal

In [3]:
def preparer(exemples):
    exemples = [ex for ex in exemples if est_valide(ex)]
    for ex in exemples:
        ex["masked_text"] = normaliser_etiquettes(ex["masked_text"])
    return exemples

validation = preparer(validation)
train = preparer(train)
train = retirer_fuite(train, validation)
print("Train après nettoyage :", len(train))
print("Validation après nettoyage :", len(validation))

Train après nettoyage : 89529
Validation après nettoyage : 22455


## 4)tirer les échantillons et vérifier

In [4]:
test = tirer_echantillon(validation, 500, graine=42)
train_v1 = tirer_echantillon(train, 5000, graine=42)

print(test[0]["source_text"])
print(test[0]["masked_text"])

18:26:13 Ilya Selhida: 'Bien sûr, Metihe Rebeca ! Je suis Tshibola Rolider et je suis impatient de voir le résultat final de notre projet de carte.'
[TIME] [GIVENNAME]: 'Bien sûr, [GIVENNAME] ! Je suis [GIVENNAME] [SURNAME] et je suis impatient de voir le résultat final de notre projet de carte.'


In [5]:
train_brut = ds_fr["train"].select_columns(colonnes).to_list()
val_brut = ds_fr["validation"].select_columns(colonnes).to_list()

invalides = [ex for ex in train_brut if not est_valide(ex)]
print("Train invalides :", len(invalides))
print("  exemples :", [ex["source_text"][:40] for ex in invalides[:5]])

train_valides = [ex for ex in train_brut if est_valide(ex)]
textes_val = {ex["source_text"] for ex in val_brut if est_valide(ex)}
fuites = [ex for ex in train_valides if ex["source_text"] in textes_val]
print("Train retirés pour fuite :", len(fuites))
print("  dont textes distincts :", len({ex["source_text"] for ex in fuites}))

Train invalides : 41
  exemples : ['...', '...', '...', '...', '-']
Train retirés pour fuite : 100
  dont textes distincts : 50


## 5) Sauvegarde

In [6]:
import json

def sauver(exemples, chemin):
    with open(chemin, "w", encoding="utf-8") as f:
        for ex in exemples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

sauver(test, "/kaggle/working/test.jsonl")
sauver(train_v1, "/kaggle/working/train_v1.jsonl")